# 23 · Teste geral: todos os acidentes de 2011 a 2025, sem recorte de tempo

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Pedido do autor (18/11, 0h58).** Além dos testes por janela de tempo, fazer a conta geral: usar todo o intervalo de acidentes de uma vez e ver se a nota funciona, sem depender da escolha de um período.

**O que é o teste geral.** A resposta de cada quilômetro é ter tido ao menos um descarrilamento por via em qualquer ano de 2011 a 2025. A nota de cada quilômetro é dada por um modelo que não viu aquele trecho: a malha é dividida em cinco grupos, quatro calibram e o quinto é conferido, e isso se repete até todos serem conferidos.

**Desenhos, declarados antes dos resultados, e de onde vêm.**

| Desenho | O que fica fora do ajuste | Base na literatura |
|---|---|---|
| G1 (principal) | trechos inteiros entre pátios, 30 sorteios | validação cruzada em blocos espaciais: quilômetros vizinhos são dependentes e precisam ficar do mesmo lado da divisão (Roberts et al., 2017) |
| G2 (mais exigente) | linhas inteiras, 10 sorteios | o mesmo, com blocos maiores |
| G0 (só referência) | quilômetros sorteados ao acaso, 10 sorteios | é a divisão ao acaso da aula 6; aqui é otimista, porque o vizinho do quilômetro conferido está no ajuste (Roberts et al., 2017) |

**Como este teste se relaciona com os testes no tempo (notebook 21).** O teste geral pergunta se a nota separa os quilômetros com e sem acidente em trechos que o modelo não viu. Os testes no tempo perguntam se a nota calibrada no passado antecipa os acidentes dos anos seguintes (Tashman, 2000; Chung e Fabbri, 2003). São perguntas diferentes; a última célula põe os resultados lado a lado.

**Gates.** 20.936 células; 2.681 descarrilamentos por via; 2.123 quilômetros atingidos; regra do EAR no período inteiro 0,588; G1 reproduz o notebook 11 (EAR calibrado 0,761, critérios contínuos 0,634, Dummy da malha 0,693; tolerância de 0,005, porque o sorteio dos grupos não é o mesmo); rótulos embaralhados dão AUC entre 0,45 e 0,55.

In [ ]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from scipy import sparse
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib; matplotlib.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})
import matplotlib.pyplot as plt
SEED = 42
C0 = pd.read_csv(f"{PROC}/11_celulas_1km.csv", low_memory=False)
C = C0[~C0["sentido"].astype(str).str.contains("Interromp")].sort_values(["ferrovia", "linha", "km_a"]).reset_index(drop=True); gate("células em operação", len(C), 20936, source="notebook 11")
E = pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E = E[E["cel_id"].isin(C["cel_id"])].copy(); E["ano"] = E["ano"].astype(int); EV = E[E["via"]]; gate("descarrilamentos por via", len(EV), 2681, source="notebook 11")
N = len(C); idx = pd.Series(np.arange(N), index=C["cel_id"].values); ANOS = np.arange(2011, 2026)
def matriz(ev):
    M_ = np.zeros((15, N))
    for k in range(15):
        s = ev[ev["ano"] == 2011 + k].groupby("cel_id").size(); M_[k, idx[s.index].values] = s.values
    return M_
CV_, CA_ = matriz(EV), matriz(E)
def conta(M_, a0, a1): return M_[(ANOS >= a0) & (ANOS <= a1)].sum(0)
EARN = ["dmin_patio", "dmin_amv", "frac_zona_patio"] + [f"f16p{j}" for j in range(5)] + [f"f16a{j}" for j in range(5)] + ["rmin_geo"] + [f"fr_lt{l}" for l in (300, 400, 500, 600, 800)]
NUM = EARN + ["rampa_max_pct", "carga_eixo_t", "taxa_dorm", "gab_vert_m", "gab_horiz_m", "cap_vinc", "dias_op", "raio_min_m", "vma_carregado"]
CAT = ["dormente", "perfil_trilho", "fixacao", "sinalizacao", "controle_circ", "bitola", "sentido", "n_linhas"]
for c in NUM: C[c] = pd.to_numeric(C[c], errors="coerce")
for c in CAT: C[c] = C[c].astype(str).str.strip().replace({"nan": "NA", "": "NA"})
def design(num, cat):
    X = C[num].copy(); med = X.median().fillna(0.0)
    for c in num: X[c + "_na"] = X[c].isna().astype(float); X[c] = X[c].fillna(med[c])
    X = pd.concat([X, pd.get_dummies(C[cat], dtype=float)], axis=1) if cat else X; return StandardScaler().fit_transform(X.values.astype(float))
X0, XE = design(NUM, CAT), design(EARN, [])
FERC = pd.factorize(C["ferrovia"])[0]; LINC = pd.factorize(C["ferrovia"].astype(str) + "|" + C["linha"].astype(str))[0]; SEGC = pd.factorize(C["seg_id"].astype(str))[0]; KM = ((C["km_a"] + C["km_b"]) / 2).values; LN = C["len_km"].values
ri, ci = [], []
for l in range(LINC.max() + 1):
    i = np.where(LINC == l)[0]; k = KM[i]; M_ = (np.abs(k[:, None] - k[None, :]) <= 20) & ~np.eye(len(i), dtype=bool); a, b = np.where(M_); ri.append(i[a]); ci.append(i[b])
W = sparse.csr_matrix((np.ones(sum(len(a) for a in ri)), (np.concatenate(ri), np.concatenate(ci))), shape=(N, N)); nviz = np.asarray(W.sum(1)).ravel()
def historico(a0, a1):
    n = a1 - a0 + 1; hv = conta(CV_, a0, a1); ha = conta(CA_, a0, a1); b = (hv > 0).astype(float); d = pd.DataFrame({"b": b, "hv": hv})
    return np.column_stack([hv / n, ha / n, b, (W @ b) / np.maximum(nviz, 1), (W @ hv) / np.maximum(nviz, 1) / n, d.groupby(SEGC)["b"].transform("mean").values, d.groupby(LINC)["b"].transform("mean").values, d.groupby(FERC)["b"].transform("mean").values, d.groupby(FERC)["hv"].transform("mean").values / n])
dp = C["dmin_patio"].fillna(1e9).values; rg = C["rmin_geo"].fillna(1e6).values
B2 = 0.5 * np.select([dp <= 300, dp <= 600, dp <= 1200, dp <= 1800], [4, 3, 2, 1], 0) / 4 + 0.5 * np.select([rg < 500, rg < 1200, rg < 2300], [3, 2, 1], 0) / 3
def auc(y, s):
    n1 = y.sum(); n0 = len(y) - n1; return float((rankdata(s)[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def auc_dentro(s, y):
    a, w = [], []
    for f in range(FERC.max() + 1):
        i = FERC == f
        if 5 <= y[i].sum() < i.sum(): a.append(auc(y[i], s[i])); w.append(y[i].sum())
    return float(np.average(a, weights=w))
def lr(Xtr, ytr, Xte): return LogisticRegression(C=0.5, max_iter=3000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
DES = np.random.default_rng(SEED).random(N)
DES = np.random.default_rng(SEED).random(N)
log_ia("23.1 dados e funções", "Claude", "FOCUS: mesmas funções do notebook 18", "design, auc, lr", "2 gates de contagem")

# 1. Teste geral: todos os descarrilamentos por via de 2011 a 2025, sem recorte de tempo
n15 = conta(CV_, 2011, 2025); y = (n15 > 0).astype(int); gate("quilômetros com descarrilamento por via em 2011 a 2025", int(y.sum()), 2123, source="notebook 11")
gate("regra do EAR no período inteiro (não depende de sorteio)", round(auc(y, B2), 3), 0.588, tol=0.002, source="notebook 11")
KF = 5
def folds_por(grupo, seed, k=KF):
    u = np.unique(grupo); f = dict(zip(u, np.random.default_rng(seed).permutation(len(u)) % k)); return np.array([f[g] for g in grupo])
def fora_do_ajuste(fo, yy):
    """Nota de cada célula dada por um modelo que não a viu: uma linha por método."""
    S = {m: np.zeros(N) for m in ("Dummy: taxa da malha", "Critérios do EAR em medida contínua", "EAR calibrado (atributos da via)")}
    for k in range(KF):
        te = fo == k; tr = ~te
        t = pd.Series(yy[tr]).groupby(FERC[tr]).mean(); S["Dummy: taxa da malha"][te] = pd.Series(FERC[te]).map(t).fillna(yy[tr].mean()).values
        S["Critérios do EAR em medida contínua"][te] = lr(XE[tr], yy[tr], XE[te]); S["EAR calibrado (atributos da via)"][te] = lr(X0[tr], yy[tr], X0[te])
    S["Regra do EAR por faixas"] = B2; return S
def topo(s, yy):
    o = np.lexsort((DES, -s)); top = np.zeros(N, bool); top[o[(np.cumsum(LN[o]) / LN.sum()) <= 0.20]] = True; return float(yy[top].sum() / yy.sum())
DESENHOS = [("G1 trechos entre pátios inteiros fora do ajuste", SEGC, 30), ("G2 linhas inteiras fora do ajuste", LINC, 10), ("G0 células sorteadas ao acaso (referência otimista)", np.arange(N), 10)]
L, GUARDA = [], {}
for nome, grupo, R in DESENHOS:
    for r in range(R):
        fo = folds_por(grupo, SEED + r)
        if not nome.startswith("G0"): assert all(len(set(fo[grupo == g])) == 1 for g in np.unique(grupo)[:50]), "grupo dividido entre ajuste e conferência"
        S = fora_do_ajuste(fo, y); GUARDA[(nome, r)] = S
        for m, s in S.items(): L.append(dict(desenho=nome, sorteio=r, metodo=m, auc=auc(y, s), auc_dentro=0.5 if m.startswith("Dummy") else auc_dentro(s, y), classe_muito_alta=topo(s, y)))
    print(nome, "concluído:", R, "sorteios", flush=True)
RES = pd.DataFrame(L); RES.to_csv(f"{TAB}/23_teste_geral_por_sorteio.csv", index=False)
T = RES.groupby(["desenho", "metodo"])[["auc", "auc_dentro", "classe_muito_alta"]].agg(["mean", "std"]).round(3); display(T); T.to_csv(f"{TAB}/23_teste_geral_resumo.csv")
m1 = RES[RES.desenho.str.startswith("G1")].groupby("metodo")["auc"].mean()
gate("G1, EAR calibrado: AUC (notebook 11, 30 sorteios)", round(float(m1["EAR calibrado (atributos da via)"]), 3), 0.761, tol=0.005, source="notebook 11")
gate("G1, critérios contínuos: AUC", round(float(m1["Critérios do EAR em medida contínua"]), 3), 0.634, tol=0.005, source="notebook 11")
gate("G1, Dummy da malha: AUC", round(float(m1["Dummy: taxa da malha"]), 3), 0.693, tol=0.005, source="notebook 11")
log_ia("23.2 teste geral", "Claude", "FOCUS: período inteiro; validação cruzada em blocos espaciais (trecho e linha); células ao acaso só como referência otimista", "23_teste_geral_resumo.csv", "5 gates contra o notebook 11")

# 2. Intervalo de 95% por reamostragem de linhas inteiras (G1, 10 primeiros sorteios, 300 reamostragens) e teste de vazamento
rng = np.random.default_rng(SEED); Lu = np.unique(LINC); ix = {l: np.where(LINC == l)[0] for l in Lu}; B = 300; nome1 = DESENHOS[0][0]
BOOT = {m: np.zeros(B) for m in ("Regra do EAR por faixas", "Dummy: taxa da malha", "Critérios do EAR em medida contínua", "EAR calibrado (atributos da via)")}; DIF = np.zeros(B)
for b in range(B):
    i = np.concatenate([ix[l] for l in rng.choice(Lu, len(Lu))])
    if y[i].sum() < 50: continue
    for m in BOOT: BOOT[m][b] = np.mean([auc(y[i], GUARDA[(nome1, r)][m][i]) for r in range(10)])
    DIF[b] = BOOT["EAR calibrado (atributos da via)"][b] - BOOT["Regra do EAR por faixas"][b]
IC = pd.DataFrame({m: dict(media=float(m1[m]), ic95_inf=float(np.percentile(v[v > 0], 2.5)), ic95_sup=float(np.percentile(v[v > 0], 97.5))) for m, v in BOOT.items()}).T.round(3); display(IC); IC.to_csv(f"{TAB}/23_teste_geral_ic95.csv")
print("EAR calibrado menos regra, IC 95%:", round(float(np.percentile(DIF[DIF != 0], 2.5)), 3), "a", round(float(np.percentile(DIF[DIF != 0], 97.5)), 3))
emb = []
for r in range(20):
    ye = np.random.default_rng(1000 + r).permutation(y); emb.append(auc(ye, fora_do_ajuste(folds_por(SEGC, SEED + r), ye)["EAR calibrado (atributos da via)"]))
gate_between("teste de vazamento (média de 20 embaralhamentos dos rótulos)", round(float(np.mean(emb)), 3), 0.45, 0.55, source="acaso")
log_ia("23.3 intervalo e vazamento", "Claude", "FOCUS: reamostragem de linhas inteiras; rótulos embaralhados devem dar 0,5", "23_teste_geral_ic95.csv", "gate de vazamento")

# 3. Os três modos de conferir, lado a lado: geral, ano a ano e três anos à frente
import os
linhas = [dict(conferencia="Geral: 2011 a 2025, trechos inteiros fora do ajuste (30 sorteios)", **{m: round(float(m1[m]), 3) for m in m1.index})]
if os.path.exists(f"{TAB}/21_validacao_temporal_por_avaliacao.csv"):
    V = pd.read_csv(f"{TAB}/21_validacao_temporal_por_avaliacao.csv"); V = V[V.desenho.str.startswith("D1")]
    for rot, nome in (("H=1", "No tempo: ano a ano (11 anos conferidos, 2015 a 2025)"), ("H=3", "No tempo: três anos à frente"), ("H=5", "No tempo: cinco anos à frente")):
        g = V[V.rotulo == rot].groupby("metodo")["auc"].mean(); linhas.append(dict(conferencia=nome, **{m: round(float(g[m]), 3) for m in m1.index if m in g.index}))
LADO = pd.DataFrame(linhas).set_index("conferencia"); display(LADO); LADO.to_csv(f"{TAB}/23_geral_e_no_tempo_lado_a_lado.csv")
exportar_registros("23")


---
## Referência

Roberts, D. R., Bahn, V., Ciuti, S., Boyce, M. S., Elith, J., Guillera-Arroita, G., Hauenstein, S., Lahoz-Monfort, J. J., Schröder, B., Thuiller, W., Warton, D. I., Wintle, B. A., Hartig, F., & Dormann, C. F. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913–929. https://doi.org/10.1111/ecog.02881 (registro conferido na Scopus em 18/11/2026; resumo lido)

## Limites

No teste geral o modelo é calibrado com acidentes de todos os anos, inclusive os mais recentes; ele mede se a nota vale para trechos não vistos, não se antecipa o futuro. Os atributos da via são os da Declaração de Rede de 2025. A AUC do país inclui a diferença entre malhas (o Dummy da malha mostra quanto); a AUC dentro da malha mede só a separação entre quilômetros da mesma malha.

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.